<a href="https://colab.research.google.com/github/juandavidr7/Taller-1-PLN/blob/main/POS_Tagging_%2B_LSTMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
juandavidrinconlopez_ancora_path = kagglehub.dataset_download('juandavidrinconlopez/ancora')
juandavidrinconlopez_conll2002_path = kagglehub.dataset_download('juandavidrinconlopez/conll2002')

print('Data source import complete.')


<p style="font-size: 2.5em; font-weight: bold;">Taller POS Tagging</p>

**Modelos:** BiLSTM | BiLSTM-Deep | BiLSTM-CRF  
**Datasets:** Ancora (español) + CoNLL2002 (español)  
**Framework:** PyTorch

# Sección 1 - Setup y Carga de Datos

## 1.1 Imports y Configuración

In [ ]:
# Instalación de la libreria pytorch-crf para el modelo con CRF
!pip install pytorch-crf

In [ ]:

import os
import json
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from itertools import product


import torch
import torch.nn as nn
from torchcrf import CRF
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Reproducibilidad
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

## 1.2 Hiperparámetros

In [ ]:
# Configuración base del modelo
CONFIG = {
    'MAX_LEN'      : 50,
    'EMBEDDING_DIM': 64,
    'HIDDEN_DIM'   : 128,
    'NUM_LAYERS'   : 1,
    'DROPOUT'      : 0.2,
    'BATCH_SIZE'   : 64,
    'LEARNING_RATE': 3e-3,
    'EPOCHS'       : 15,
    'MIN_FREQ'     : 3,
    'PAD_IDX'      : 0,
    'UNK_IDX'      : 1,
}

# Grid de hiperparámetros para búsqueda
HYPERPARAMS = {
    'batch_size'   : [16, 32, 64],
    'embedding_dim': [100, 300],
    'hidden_dim'   : [128, 256],
    'optimizer'    : ['adam', 'sgd']
}

MAX_EPOCHS = 50
PATIENCE   = 5

print('CONFIG:')
for k, v in CONFIG.items():
    print(f'  {k:<18}: {v}')

## 1.3 Carga del Dataset CoNLL2002 y Ancora

In [ ]:
ANCORA_PATH      = '/kaggle/input/private-dataset/ancora_corpus_pos (1).csv'
CONLL_TRAIN_PATH = '/kaggle/input/datasets/juandavidrinconlopez/conll2002/train.txt'
CONLL_VALID_PATH = '/kaggle/input/datasets/juandavidrinconlopez/conll2002/valid.txt'
CONLL_TEST_PATH  = '/kaggle/input/datasets/juandavidrinconlopez/conll2002/test.txt'

def load_ancora(path: str) -> pd.DataFrame:
    """
    Carga Ancora desde CSV.
    Columnas: Sentence # | Word | Tag (NER) | POS
    """
    df = pd.read_csv(path, encoding='utf-8')
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={
        'Sentence #': 'sentence_id',
        'Word'      : 'word',
        'Tag'       : 'ner',
        'POS'       : 'pos'
    })
    df['sentence_id'] = df['sentence_id'].ffill()
    df = df.dropna(subset=['word', 'pos']).reset_index(drop=True)
    df['word'] = df['word'].astype(str)
    df['pos']  = df['pos'].astype(str)
    return df


def load_conll2002(path: str) -> pd.DataFrame:
    """
    Carga CoNLL2002 desde archivo de texto plano.
    Formato por línea: WORD POS NER
    Oraciones separadas por líneas vacías.
    """
    records = []
    sentence_id = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if line.strip() == '':
                sentence_id += 1
                continue
            if line.startswith('-DOCSTART-'):
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            records.append({
                'sentence_id': sentence_id,
                'word'       : str(parts[0]),
                'pos'        : str(parts[1]),
                'ner'        : str(parts[2]) if len(parts) > 2 else 'O'
            })
    return pd.DataFrame(records)


# Cargar
ancora_df      = load_ancora(ANCORA_PATH)
conll_train_df = load_conll2002(CONLL_TRAIN_PATH)
conll_val_df   = load_conll2002(CONLL_VALID_PATH)
conll_test_df  = load_conll2002(CONLL_TEST_PATH)

print(f'Ancora         : {len(ancora_df):,} tokens | {ancora_df["sentence_id"].nunique():,} oraciones')
print(f'CoNLL train    : {len(conll_train_df):,} tokens | {conll_train_df["sentence_id"].nunique():,} oraciones')
print(f'CoNLL val      : {len(conll_val_df):,} tokens | {conll_val_df["sentence_id"].nunique():,} oraciones')
print(f'CoNLL test     : {len(conll_test_df):,} tokens | {conll_test_df["sentence_id"].nunique():,} oraciones')

## 1.4 Observación y comparativa entre Datasets

In [ ]:
# Distribución de tags y longitudes
tag_counts_ancora = ancora_df['pos'].value_counts()
tag_counts_conll  = conll_train_df['pos'].value_counts()
sent_lengths_ancora = ancora_df.groupby('sentence_id')['word'].count()
sent_lengths_conll  = conll_train_df.groupby('sentence_id')['word'].count()

print('=== Ancora ===')
print(f'  Tags únicos : {ancora_df["pos"].nunique()}')
print(f'  Longitud P95: {sent_lengths_ancora.quantile(0.95):.0f}')
print('\n=== CoNLL2002 Train ===')
print(f'  Tags únicos : {conll_train_df["pos"].nunique()}')
print(f'  Longitud P95: {sent_lengths_conll.quantile(0.95):.0f}')

# Comparativa
summary = pd.DataFrame({
    'Métrica': ['Total tokens','Total oraciones','Tags únicos','Longitud media','P95 longitud'],
    'Ancora' : [
        f'{len(ancora_df):,}',
        f'{ancora_df["sentence_id"].nunique():,}',
        ancora_df['pos'].nunique(),
        f'{sent_lengths_ancora.mean():.1f}',
        f'{sent_lengths_ancora.quantile(0.95):.0f}'
    ],
    'CoNLL Train': [
        f'{len(conll_train_df):,}',
        f'{conll_train_df["sentence_id"].nunique():,}',
        conll_train_df['pos'].nunique(),
        f'{sent_lengths_conll.mean():.1f}',
        f'{sent_lengths_conll.quantile(0.95):.0f}'
    ]
})
print('\n', summary.to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Ancora — Top tags
top20a = tag_counts_ancora.head(20)
axes[0,0].barh(top20a.index[::-1], top20a.values[::-1], color='steelblue')
axes[0,0].set_title('Top-20 POS Tags — Ancora')
axes[0,0].set_xlabel('Frecuencia')

# Ancora — Longitudes
axes[0,1].hist(sent_lengths_ancora, bins=40, color='darkorange', edgecolor='white')
axes[0,1].axvline(sent_lengths_ancora.quantile(0.95), color='red', linestyle='--',
                  label=f'P95={sent_lengths_ancora.quantile(0.95):.0f}')
axes[0,1].set_title('Longitudes — Ancora')
axes[0,1].legend()

# CoNLL — Top tags
top20c = tag_counts_conll.head(20)
axes[1,0].barh(top20c.index[::-1], top20c.values[::-1], color='seagreen')
axes[1,0].set_title('Top-20 POS Tags — CoNLL2002')
axes[1,0].set_xlabel('Frecuencia')

# CoNLL — Longitudes
axes[1,1].hist(sent_lengths_conll, bins=40, color='mediumpurple', edgecolor='white')
axes[1,1].axvline(sent_lengths_conll.quantile(0.95), color='red', linestyle='--',
                  label=f'P95={sent_lengths_conll.quantile(0.95):.0f}')
axes[1,1].set_title('Longitudes — CoNLL2002')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('exploracion.png', dpi=120)
plt.show()

# Sección 2 - Preprocesamiento de los Datos

 ## 2.1 Armado de Frases y Vocabulario

In [ ]:
# ── Paso 1: Construcción de oraciones ────────────────────────────────────────
def df_to_sentences(df, max_len):
    """DataFrame → listas paralelas de words y tags, truncadas a max_len."""
    sentences, pos_seqs = [], []
    for _, group in df.groupby('sentence_id', sort=False):
        sentences.append(group['word'].tolist()[:max_len])
        pos_seqs.append(group['pos'].tolist()[:max_len])
    return sentences, pos_seqs

MAX_LEN = CONFIG['MAX_LEN']

ancora_sents,      ancora_tags       = df_to_sentences(ancora_df,      MAX_LEN)
conll_train_sents, conll_train_tags  = df_to_sentences(conll_train_df, MAX_LEN)
conll_val_sents,   conll_val_tags    = df_to_sentences(conll_val_df,   MAX_LEN)
conll_test_sents,  conll_test_tags   = df_to_sentences(conll_test_df,  MAX_LEN)

print(f'Ancora  → {len(ancora_sents):,} oraciones')
print(f'CoNLL   → train: {len(conll_train_sents):,} | val: {len(conll_val_sents):,} | test: {len(conll_test_sents):,}')

# ── Paso 2: Vocabulario e indexación ─────────────────────────────────────────
def build_vocab(sentences, tags, min_freq=3):
    """Construye word2idx y tag2idx reproducibles."""
    word_counts = Counter(w for sent in sentences for w in sent)
    word2idx = {'<PAD>': 0, '<UNK>': 1}
    for word, count in word_counts.most_common():
        if count >= min_freq:
            word2idx[word] = len(word2idx)

    all_tags = sorted(set(t for seq in tags for t in seq))
    tag2idx  = {'<PAD>': 0}
    for tag in all_tags:
        tag2idx[tag] = len(tag2idx)

    idx2word = {v: k for k, v in word2idx.items()}
    idx2tag  = {v: k for k, v in tag2idx.items()}
    return word2idx, tag2idx, idx2word, idx2tag

# Vocabularios independientes por corpus
ancora_w2i, ancora_t2i, ancora_i2w, ancora_i2t = build_vocab(
    ancora_sents, ancora_tags, CONFIG['MIN_FREQ']
)
conll_w2i, conll_t2i, conll_i2w, conll_i2t = build_vocab(
    conll_train_sents, conll_train_tags, CONFIG['MIN_FREQ']
)

print(f'Ancora  → palabras: {len(ancora_w2i):,} | tags: {len(ancora_t2i)}')
print(f'CoNLL   → palabras: {len(conll_w2i):,}  | tags: {len(conll_t2i)}')
print(f'\nTags Ancora : {list(ancora_t2i.keys())}')
print(f'Tags CoNLL  : {list(conll_t2i.keys())}')

## 2.2 Vectorización y padding

In [ ]:
# ── Paso 5: Vectorización + Padding ──────────────────────────────────────────
def vectorizar(sentences, pos_tags, word2idx, tag2idx, max_len):
    """Convierte strings a arrays numéricos con padding. Guarda lengths reales."""
    print('🔢 Vectorizando...')
    start = time.time()
    n = len(sentences)
    X = np.zeros((n, max_len), dtype=np.int32)
    y = np.zeros((n, max_len), dtype=np.int32)
    lengths = np.zeros(n, dtype=np.int32)
    for i, (words, tags) in enumerate(zip(sentences, pos_tags)):
        seq_len = min(len(words), max_len)
        lengths[i] = seq_len
        X[i, :seq_len] = [word2idx.get(w, 1) for w in words[:seq_len]]
        y[i, :seq_len] = [tag2idx.get(t, 0)  for t in tags[:seq_len]]
    print(f'   Tiempo: {time.time()-start:.2f}s | Forma: {X.shape}')
    return X, y, lengths

# Ancora
X_ancora, y_ancora, len_ancora = vectorizar(
    ancora_sents, ancora_tags, ancora_w2i, ancora_t2i, MAX_LEN
)
# CoNLL
X_ctr, y_ctr, len_ctr = vectorizar(conll_train_sents, conll_train_tags, conll_w2i, conll_t2i, MAX_LEN)
X_cva, y_cva, len_cva = vectorizar(conll_val_sents,   conll_val_tags,   conll_w2i, conll_t2i, MAX_LEN)
X_cte, y_cte, len_cte = vectorizar(conll_test_sents,  conll_test_tags,  conll_w2i, conll_t2i, MAX_LEN)

## 2.3 Split de los dataset en train, val y test

In [ ]:
# ── Paso 3 y 4: Split Ancora 70/15/15 ────────────────────────────────────────
X_atr, X_atemp, y_atr, y_atemp = train_test_split(
    X_ancora, y_ancora, test_size=0.30, random_state=SEED
)
X_ava, X_ate, y_ava, y_ate = train_test_split(
    X_atemp, y_atemp, test_size=0.50, random_state=SEED
)

print('=== Splits Ancora ===')
print(f'  Train : {X_atr.shape[0]:,}')
print(f'  Val   : {X_ava.shape[0]:,}')
print(f'  Test  : {X_ate.shape[0]:,}')
print('\n=== Splits CoNLL2002 (nativos) ===')
print(f'  Train : {X_ctr.shape[0]:,}')
print(f'  Val   : {X_cva.shape[0]:,}')
print(f'  Test  : {X_cte.shape[0]:,}')

# Sección 3 - Definición de modelos y data loaders

## 3.1 Dataloaders (especificos de Pytorch)

In [ ]:
# ── DataLoaders
def create_dataloaders(X_train, y_train, X_val, y_val, batch_size):
    """Crea DataLoaders listos para entrenar BiLSTMDeep."""
    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.long),
        torch.tensor(y_train, dtype=torch.long)
    )
    val_ds = TensorDataset(
        torch.tensor(X_val, dtype=torch.long),
        torch.tensor(y_val, dtype=torch.long)
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size)
    return train_loader, val_loader

# Ancora
ancora_train_loader, ancora_val_loader = create_dataloaders(
    X_atr, y_atr, X_ava, y_ava, CONFIG['BATCH_SIZE']
)
# CoNLL
conll_train_loader, conll_val_loader = create_dataloaders(
    X_ctr, y_ctr, X_cva, y_cva, CONFIG['BATCH_SIZE']
)

# Verificación
xb, yb = next(iter(ancora_train_loader))
print(f'Batch Ancora — X: {xb.shape} | y: {yb.shape}')

## 3.2 BiLSTM

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim, hidden_dim):
        super(BiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.bilstm    = nn.LSTM(
            embedding_dim, hidden_dim,
            num_layers=1, bidirectional=True,
            batch_first=True, dropout=0.3
        )
        self.fc = nn.Linear(hidden_dim*2, tagset_size)

    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.bilstm(x)
        return self.fc(lstm_out)

## 3.3 BiLSTM - Deep

In [ ]:
class BiLSTMDeep(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim, hidden_dim):
        super(BiLSTMDeep, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.bilstm    = nn.LSTM(
            embedding_dim, hidden_dim,
            num_layers=2, bidirectional=True,
            batch_first=True, dropout=0.3
        )
        self.fc = nn.Linear(hidden_dim, tagset_size)

    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.bilstm(x)
        return self.fc(lstm_out)

## 3.4 BiLSTM - CRF

In [ ]:
class BiLSTMCRF(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim, hidden_dim):
        super(BiLSTMCRF, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.bilstm    = nn.LSTM(
            embedding_dim, hidden_dim,
            num_layers=1, bidirectional=True,
            batch_first=True
        )
        # Capa de dropout explícita
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim*2, tagset_size)
        self.CRF = CRF(tagset_size, batch_first=True)

    def forward(self, x, tags=None):
        # 1. Creación de la máscara
        # Esto genera un tensor de True donde hay palabras y False donde hay relleno
        mask = (x != 0).to(x.device)

        #Flujo normal
        x = self.embedding(x)
        lstm_out, _ = self.bilstm(x)

        # Aplicación del dropout manualmente
        lstm_out = self.dropout(lstm_out)

        # Emisiones (logits) para cada etiqueta
        emissions = self.fc(lstm_out)

        if tags is not None:
            # Modo entrenamiento: calcula la pérdida
            log_likelihood = self.CRF(emissions, tags, mask=mask, reduction='mean')

            # Se usa el negativo porque los optimizadores de PyTorch MINIMIZAN funciones
            return -log_likelihood

        else:
            # Inferencia (predicción): Retorna la mejor secuencia de etiquetas usando el algoritmo de Viterbi
            return self.CRF.decode(emissions, mask=mask)

# Sección 4 - Entrenamiento y evaluación

## 4.1 Entrenamiento

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, criterion):
    """Entrenamiento con Early Stopping (patience=5)."""
    best_val_loss     = float('inf')
    patience_counter  = 0
    best_model_state  = None

    for epoch in range(MAX_EPOCHS):
        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        train_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            #--- IMPORTANTE: Detección del modelo---
            if isinstance(model,BiLSTMCRF):
                # El modelo CRF calcula su propia pérdida internamente
                loss = model(inputs, tags=labels)
            else:
                # Los modelos anteriores usan la lógica que se adapta a ellos
                outputs = model(inputs)
                loss = criterion(outputs.view(-1, outputs.shape[-1]), labels.view(-1))

            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # ── Validación ─────────────────────────────────────────────────────
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                #--- IMPORTANTE: Detección del modelo---
                if isinstance(model, BiLSTMCRF):
                    loss = model(inputs, tags=labels)
                else:
                    outputs = model(inputs)
                    loss = criterion(outputs.view(-1, outputs.shape[-1]), labels.view(-1))

                val_loss += loss.item()
        val_loss /= len(val_loader)

        print(f'Epoch {epoch+1:>3} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

        # ── Early Stopping ─────────────────────────────────────────────────
        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'Early stopping en época {epoch+1}')
                break

    # Restaurar mejor modelo
    if best_model_state:
        model.load_state_dict(best_model_state)

    return best_val_loss

def execute_train(X_train, y_train, X_val, y_val, word2idx, tag2idx, model_class=BiLSTMDeep):
    """Grid search sobre hiperparámetros."""
    results = []
    best_model  = None
    best_config = None
    best_loss   = float('inf')

    combos = list(product(
        HYPERPARAMS['batch_size'],
        HYPERPARAMS['embedding_dim'],
        HYPERPARAMS['hidden_dim'],
        HYPERPARAMS['optimizer']
    ))
    print(f'Total combinaciones: {len(combos)}')

    for batch_size, emb_dim, hid_dim, opt in combos:
        print(f'\n{"="*50}')
        print(f'Batch:{batch_size} | Emb:{emb_dim} | Hidden:{hid_dim} | Opt:{opt}')
        print('='*50)

        train_loader, val_loader = create_dataloaders(
            X_train, y_train, X_val, y_val, batch_size
        )
        model = model_class(
            vocab_size=len(word2idx),
            tagset_size=len(tag2idx),
            embedding_dim=emb_dim,
            hidden_dim=hid_dim
        ).to(device)

        criterion = nn.CrossEntropyLoss(ignore_index=0)
        optimizer = (
            torch.optim.Adam(model.parameters(), lr=0.001)
            if opt == 'adam'
            else torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
        )

        val_loss = train_model(model, train_loader, val_loader, optimizer, criterion)

        results.append({
            'batch': batch_size, 'embedding': emb_dim,
            'hidden': hid_dim, 'optimizer': opt, 'val_loss': val_loss
        })

        if val_loss < best_loss:
            best_loss   = val_loss
            best_model  = model
            best_config = {'batch': batch_size, 'embedding': emb_dim,
                           'hidden': hid_dim, 'optimizer': opt}

    print(f'\n✅ Mejor config: {best_config} | Val Loss: {best_loss:.4f}')
    return pd.DataFrame(results), best_model

## 4.2 Evaluación

In [ ]:
def evaluar(model, X_test, y_test, idx2tag, batch_size=128):
    """Evaluación con classification_report ignorando PAD."""
    model.eval()
    test_ds     = TensorDataset(
        torch.tensor(X_test, dtype=torch.long),
        torch.tensor(y_test, dtype=torch.long)
    )
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)

            # --- IMPORTANTE: CASOS SEGÚN EL MODELO---
            if isinstance(outputs, list):
                # Caso CRF: la salida ya es la mejor secuencia (lista de listas)
                preds = outputs
            else:
                # Caso BiLSTM/Deep: la salida es un Tensor de logits
                preds = outputs.argmax(dim=-1).cpu().numpy()
            # -----------------------------------------
            labels = labels.numpy()
            for i in range(len(inputs)):
                seq_len = int(np.sum(inputs[i].cpu().numpy() != 0))

                # Extraer predicciones según el formato
                if isinstance (preds, list):
                    # Para CRF: toma la sublista directamente
                    current_preds = preds[i]
                else:
                    # Para BiLSTM: toma la rebanada segun el array
                    current_preds = preds[i, :seq_len]

                all_preds.extend([idx2tag[p] for p in current_preds[:seq_len]])
                all_labels.extend([idx2tag[l] for l in labels[i, :seq_len]])

    acc    = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, zero_division=0)
    print(f'Accuracy: {acc*100:.2f}%')
    print(report)
    return acc, report

In [ ]:
def predecir(model, sentence, word2idx, idx2tag, max_len=50):
    """Predice POS tags para una oración en texto libre."""
    words    = sentence.strip().split()
    if not words:
        return [], []
    word_ids = [word2idx.get(w, 1) for w in words]
    word_ids += [0] * (max_len - len(word_ids))
    tensor   = torch.tensor([word_ids], dtype=torch.long).to(device)

    model.eval()
    with torch.no_grad():
        outputs   = model(tensor)
        # --- IMPORTANTE: CASOS SEGÚN EL MODELO ---
        if isinstance(outputs, list):
            # Caso CRF: Ya es una lista de listas, se toma la primera frase
            pred_ids = outputs[0]
        else:
            # Caso BiLSTM/Deep: Es un tensor, se usa el argmax original
            pred_ids = outputs.argmax(dim=-1).squeeze(0).cpu().numpy()
        # -----------------------------------------
    tags = [idx2tag[int(p)] for p in pred_ids[:len(words)]]
    return words, tags


def modo_interactivo(model, word2idx, idx2tag, max_len=50):
    """Modo interactivo para probar el modelo con oraciones propias."""
    print('\n' + '='*60)
    print('🎯 MODO INTERACTIVO — POS TAGGING')
    print('='*60)
    print("Escribe 'salir' para terminar\n")
    while True:
        try:
            sentence = input('✍️  Oración: ').strip()
            if sentence.lower() in ['salir', 'quit', 'exit', 'q']:
                print('\n👋 ¡Hasta luego!\n')
                break
            if not sentence:
                continue
            words, tags = predecir(model, sentence, word2idx, idx2tag, max_len)
            print(f'\n{"Palabra":<20} {"Tag":<10}')
            print('-' * 30)
            for w, t in zip(words, tags):
                print(f'{w:<20} {t:<10}')
            print()
        except KeyboardInterrupt:
            print('\n👋')
            break

# Sección 5 - Ejecución de modelos

## 5.1 modelo BiLSTM

In [ ]:
"""def ancora_grid_search_BiLSTM():
    print('='*60)
    print('🚀 ANCORA — Grid Search BiLSTM')
    print('='*60)

    results_ancora, best_model_ancora = execute_train(
        X_atr, y_atr, X_ava, y_ava,
        ancora_w2i, ancora_t2i,
        model_class=BiLSTM
    )

    print('\n📊 Resultados grid search Ancora:')
    print(results_ancora.sort_values('val_loss').to_string(index=False))

    print('\n📋 Evaluación sobre test Ancora:')
    evaluar(best_model_ancora, X_ate, y_ate, ancora_i2t)

    return best_model_ancora

def conll_grid_search_BiLSTM():
    print('='*60)
    print('🚀 CoNLL2002 — Grid Search BiLSTM')
    print('='*60)

    results_conll, best_model_conll = execute_train(
        X_ctr, y_ctr, X_cva, y_cva,
        conll_w2i, conll_t2i,
        model_class=BiLSTM
    )

    print('\n📊 Resultados grid search CoNLL:')
    print(results_conll.sort_values('val_loss').to_string(index=False))

    print('\n📋 Evaluación sobre test CoNLL:')
    evaluar(best_model_conll, X_cte, y_cte, conll_i2t)

    return best_model_conll

best_model_ancora =  ancora_grid_search_BiLSTM()
"""


In [ ]:
#best_model_conll = conll_grid_search_BiLSTM()

In [ ]:
#modo_interactivo(best_model_ancora, ancora_w2i, ancora_i2t, MAX_LEN)

In [ ]:
#modo_interactivo(best_model_conll, conll_w2i, conll_i2t, MAX_LEN)

## 5.2 modelo BiLSTM-Deep


In [ ]:

"""def main_BiLSTM_deep():
    print('='*60)
    print('🚀 ANCORA — Grid Search BiLSTMDeep')
    print('='*60)

    results_ancora, best_model_ancora = execute_train(
        X_atr, y_atr, X_ava, y_ava,
        ancora_w2i, ancora_t2i,
        model_class=BiLSTMDeep
    )

    print('\n📊 Resultados grid search Ancora:')
    print(results_ancora.sort_values('val_loss').to_string(index=False))

    print('\n📋 Evaluación sobre test Ancora:')
    evaluar(best_model_ancora, X_ate, y_ate, ancora_i2t)

    print('='*60)
    print('🚀 CoNLL2002 — Grid Search BiLSTMDeep')
    print('='*60)

    results_conll, best_model_conll = execute_train(
        X_ctr, y_ctr, X_cva, y_cva,
        conll_w2i, conll_t2i,
        model_class=BiLSTMDeep
    )

    print('\n📊 Resultados grid search CoNLL:')
    print(results_conll.sort_values('val_loss').to_string(index=False))

    print('\n📋 Evaluación sobre test CoNLL:')
    evaluar(best_model_conll, X_cte, y_cte, conll_i2t)

    return best_model_ancora, best_model_conll

best_model_ancora, best_model_conll =  main_BiLSTM_deep()
"""


In [ ]:
#modo_interactivo(best_model_ancora, ancora_w2i, ancora_i2t, MAX_LEN)

In [ ]:
#modo_interactivo(best_model_conll, conll_w2i, conll_i2t, MAX_LEN)

## 5.3 modelo BiLSTM - CRF

In [ ]:
def main_BiLSTM_CRF():
    print('='*60)
    print('🚀 ANCORA — Grid Search BiLSTM + CRF')
    print('='*60)

    # Pasamos BiLSTMCRF como la clase del modelo
    results_ancora, best_model_ancora = execute_train(
        X_atr, y_atr, X_ava, y_ava,
        ancora_w2i, ancora_t2i,
        model_class=BiLSTMCRF
    )

    print('\n📊 Resultados grid search Ancora (CRF):')
    print(results_ancora.sort_values('val_loss').to_string(index=False))

    print('\n📋 Evaluación sobre test Ancora (CRF):')
    evaluar(best_model_ancora, X_ate, y_ate, ancora_i2t)

    print('='*60)
    print('🚀 CoNLL2002 — Grid Search BiLSTM + CRF')
    print('='*60)

    results_conll, best_model_conll = execute_train(
        X_ctr, y_ctr, X_cva, y_cva,
        conll_w2i, conll_t2i,
        model_class=BiLSTMCRF
    )

    print('\n📊 Resultados grid search CoNLL (CRF):')
    print(results_conll.sort_values('val_loss').to_string(index=False))

    print('\n📋 Evaluación sobre test CoNLL (CRF):')
    evaluar(best_model_conll, X_cte, y_cte, conll_i2t)

    return best_model_ancora, best_model_conll

# Ejecutar el experimento con CRF
best_model_ancora_crf, best_model_conll_crf = main_BiLSTM_CRF()


In [ ]:
# Probar el modelo BiLSTM-CRF entrenado con el dataset Ancora
#modo_interactivo(best_model_ancora_crf, ancora_w2i, ancora_i2t, MAX_LEN)


In [ ]:
# Probar el modelo BiLSTM-CRF entrenado con CoNLL2002
#modo_interactivo(best_model_conll_crf, conll_w2i, conll_i2t, MAX_LEN)


# Extras:
## Guardar modelos:


In [ ]:
#guardado de modelo:
# Guardar el modelo completo
#Guarda dos archivos el archivo con los pesos PTH y un archivo pickle con el vocabulario necesario
def save_all_model(model, word2idx, idx2tag, name_dataset, name_model):
    archivo_pesos =  f'modelo_{name_model}_completo.pth'
    archivo_vocabs = f'vocabularios_{name_dataset}.pkl'
    torch.save(model, archivo_pesos)


    with open(archivo_vocabs, 'wb') as f:
        pickle.dump({'w2i': word2idx, 'i2t': idx2tag}, f)
    return os.path.abspath(archivo_pesos), os.path.abspath(archivo_vocabs)


def load_all_model(path_pth, path_vocal_pkl):
    with open(path_vocal_pkl, 'rb') as f:
        vocabs = pickle.load(f)
        w2i = vocabs['w2i']
        i2t = vocabs['i2t']

    # Cargar el modelo completo directamente
    return torch.load(path_pth, weights_only=False), w2i, i2t

#Ejemplo de uso
'''
    #Guardado retorna las rutas donde se guardo (recomendable guardarlo en local por si se borra la cache no se xd).
    path_plt,path_vocal_pkl = save_all_model(best_model_ancora, ancora_w2i, ancora_i2t,"ancora", "BiLSTM-Deep")

    print(path_plt,path_vocal_pkl)

    #Carga del modelo retorna el modelo y los elementos del vocabulario
    model_test, testw2i, testi2t = load_all_model(path_plt,path_vocal_pkl)

    #Uso de modelo de prueba
    modo_interactivo(model_test, testw2i, testi2t, MAX_LEN)
'''

In [ ]:
#save_all_model(best_model_ancora, ancora_w2i, ancora_i2t, "vocabularioAncoraBiLSTM", "ancoraBiLSTM")

#save_all_model(best_model_conll, conll_w2i, conll_i2t, "vocabularioConllBiLSTM", "conllBiLSTM")

save_all_model(best_model_ancora_crf, ancora_w2i, ancora_i2t, "vocabularioAncoraBiLSTMCRF", "ancoraBiLSTMCRF")
save_all_model(best_model_conll_crf, conll_w2i, conll_i2t, "vocabularioConllBiLSTMCRF", "conllBiLSTMCRF")